# CartPole PyBullet Classification

This notebook trains a classifier to predict ROA membership for CartPole PyBullet trajectories.

**Key techniques:**
- Transforms `theta` to `sin(theta)` and `cos(theta)` (5 features: [x, sin(theta), cos(theta), x_dot, theta_dot])
- Only normalizes features [0, 3, 4] (x, x_dot, theta_dot), NOT sin/cos features
- Uses balanced training with class weights and WeightedRandomSampler

**All parameters are loaded from config file - no hardcoded values!**

In [1]:
import sys
sys.path.append('../..')
from src.utils.notebook_config import NotebookConfig

# Load ALL parameters from config file
cfg = NotebookConfig('cartpole_pybullet')

print("✓ Configuration loaded!")
print(f"  To change parameters, edit: configs/notebook/cartpole_pybullet_viz.yaml")


✓ Configuration loaded!
  To change parameters, edit: configs/notebook/cartpole_pybullet_viz.yaml


## Step 2: Import Libraries


In [ ]:
# Import libraries
missing_packages = []
try:
    import numpy as np
    print(f"✓ NumPy {np.__version__}")
except ImportError:
    missing_packages.append("numpy")
    print("✗ NumPy not installed")

try:
    import torch
    print(f"✓ PyTorch {torch.__version__}")
    print(f"  CUDA available: {torch.cuda.is_available()}")
except ImportError:
    missing_packages.append("torch")
    print("✗ PyTorch not installed")

try:
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
    import sklearn
    print(f"✓ scikit-learn {sklearn.__version__}")
except ImportError:
    missing_packages.append("scikit-learn")
    print("✗ scikit-learn not installed")

try:
    import matplotlib.pyplot as plt
    print(f"✓ matplotlib")
except ImportError:
    missing_packages.append("matplotlib")
    print("✗ matplotlib not installed")

if missing_packages:
    print(f"\n⚠️  MISSING PACKAGES: {', '.join(missing_packages)}")
    print("\nTo install, run this in a terminal:")
    print(f"  pip install {' '.join(missing_packages)}")
    raise ImportError(f"Please install missing packages: {', '.join(missing_packages)}")

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

print("\n✓ All libraries imported successfully!")


## Step 3: Load Training and Validation Data

In [ ]:
# Load data using config parameters
from src.data.cartpole_pybullet_data import CartPolePyBulletDataModule

# Get parameters from config
train_size = cfg['data.train_size']
val_size = cfg['data.val_size']
data_dir = cfg['data.data_dir']

# Create data module
datamodule = CartPolePyBulletDataModule(
    data_file=cfg['data.roa_labels_file'],
    data_dir=data_dir,
    train_size=train_size,
    val_size=val_size,
    batch_size=cfg['training.batch_size'],
    num_workers=0,  # Notebooks typically use 0 workers
    use_weighted_sampler=True,
    pin_memory=False
)

# Setup and load data
datamodule.setup()

# Extract data from datamodule
X_train = datamodule.train_dataset.features.numpy()
y_train = datamodule.train_dataset.labels.numpy()
X_val = datamodule.val_dataset.features.numpy()
y_val = datamodule.val_dataset.labels.numpy()
feature_max = datamodule.feature_max

print(f"\n✓ Data loaded successfully!")
print(f"  Training: {len(X_train)} datapoints")
print(f"  Validation: {len(X_val)} datapoints")
print(f"  Features: {X_train.shape[1]} (x, sin(theta), cos(theta), x_dot, theta_dot)")
print(f"  Note: theta transformed to sin/cos, only features [0,3,4] normalized")


## Step 4: Calculate Class Weights


In [ ]:
# Calculate class weights for balanced training
def calculate_class_weights(y):
    """
    Calculate class weights for balanced training.
    Returns weights that can be used with WeightedRandomSampler or in loss function.
    Formula: weight_i = total_samples / (num_classes × count_of_class_i)
    """
    unique, counts = np.unique(y, return_counts=True)
    total = len(y)
    num_classes = len(unique)
    
    # Calculate inverse frequency weights
    class_weights = {}
    for cls, count in zip(unique, counts):
        weight = total / (num_classes * count)
        class_weights[int(cls)] = weight
    
    # Create sample weights array (weight for each sample based on its class)
    sample_weights = np.array([class_weights[int(label)] for label in y], dtype=np.float32)
    
    return class_weights, sample_weights

# Calculate weights for training set
class_weights, sample_weights = calculate_class_weights(y_train)

print("Class weights for balanced training:")
print(f"  Class 0 (failure): {class_weights[0]:.4f}")
print(f"  Class 1 (success): {class_weights[1]:.4f}")
print(f"\nClass distribution:")
print(f"  Failures: {np.sum(y_train == 0)} ({np.sum(y_train == 0)/len(y_train):.2%})")
print(f"  Successes: {np.sum(y_train == 1)} ({np.sum(y_train == 1)/len(y_train):.2%})")

# Convert to tensor for loss function
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class_weights_tensor = torch.FloatTensor([class_weights[0], class_weights[1]]).to(device)


## Step 5: Create Datasets and DataLoaders


In [ ]:
# Define Dataset class for timestep-level data
class TimestepDataset(Dataset):
    def __init__(self, X, y):
        self.X = X.astype(np.float32)
        self.y = y.astype(np.int64)  # Use int64 for CrossEntropyLoss
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return torch.FloatTensor(self.X[idx]), torch.LongTensor([self.y[idx]]).squeeze()

# Create datasets
train_dataset = TimestepDataset(X_train, y_train)
val_dataset = TimestepDataset(X_val, y_val)

print(f"Training dataset size: {len(train_dataset)} timesteps")
print(f"Validation dataset size: {len(val_dataset)} timesteps")

# Create weighted sampler for balanced training
weighted_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# Get batch size from config
batch_size = cfg['training.batch_size']

# Create data loaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    sampler=weighted_sampler,  # Use weighted sampler instead of shuffle
    num_workers=0
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=batch_size, 
    shuffle=False,
    num_workers=0
)

print(f"\nBatch size: {batch_size}")
print(f"Training batches per epoch: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")


## Step 6: Initialize Model


In [ ]:
# Get model parameters from config
from src.models.classifier import Classifier

input_dim = cfg['model.input_dim']
hidden_dims = cfg['model.hidden_dims']
output_dim = cfg['model.output_dim']
dropout = cfg['model.dropout']

# Initialize model
model = Classifier(
    input_dim=input_dim,
    hidden_dims=hidden_dims,
    output_dim=output_dim,
    dropout=dropout,
    use_batch_norm=True
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model initialized on: {device}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:")
print(f"  Input dim: {input_dim}")
print(f"  Hidden dims: {hidden_dims}")
print(f"  Output dim: {output_dim}")
print(f"  Dropout: {dropout}")


## Step 7: Setup Loss and Optimizer


In [ ]:
# Get learning rate from config
learning_rate = cfg['training.learning_rate']

# Define loss function with class weights and optimizer
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print(f"Loss function: CrossEntropyLoss with class weights")
print(f"  Class 0 weight: {class_weights_tensor[0]:.4f}")
print(f"  Class 1 weight: {class_weights_tensor[1]:.4f}")
print(f"Optimizer: Adam with learning rate {learning_rate}")


## Step 8: Training and Validation Functions


In [ ]:
# Training and validation functions
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())
    
    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / total
    
    # Calculate F1 score
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    f1 = f1_score(all_labels, all_preds)
    
    return avg_loss, accuracy, f1

def validate(model, val_loader, criterion, device):
    """Validate the model"""
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())
    
    avg_loss = total_loss / len(val_loader)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    
    return avg_loss, accuracy, precision, recall, f1, all_preds, all_labels

print("Training functions defined!")


## Step 9: Training Loop


In [ ]:
# Get number of epochs from config (or use default)
num_epochs = cfg.get('training.num_epochs', 10)

train_losses = []
train_accuracies = []
train_f1_scores = []
val_losses = []
val_accuracies = []
val_precisions = []
val_recalls = []
val_f1_scores = []

best_val_f1 = 0
best_model_state = None

print("=" * 60)
print("Starting training...")
print("=" * 60)
print()

for epoch in range(num_epochs):
    # Train
    train_loss, train_acc, train_f1 = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc, val_prec, val_rec, val_f1, val_preds, val_labels = validate(
        model, val_loader, criterion, device
    )
    
    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = model.state_dict().copy()
    
    # Store metrics
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    train_f1_scores.append(train_f1)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    val_precisions.append(val_prec)
    val_recalls.append(val_rec)
    val_f1_scores.append(val_f1)
    
    # Print progress
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}]")
        print(f"  Train - Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%, F1: {train_f1:.4f}")
        print(f"  Val   - Loss: {val_loss:.4f}, Acc: {val_acc*100:.2f}%, F1: {val_f1:.4f}")
        print(f"         Precision: {val_prec:.4f}, Recall: {val_rec:.4f}")
        print()

# Load best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"✓ Loaded best model (Validation F1: {best_val_f1:.4f})")


## Step 10: Final Validation Results


In [ ]:
# Final evaluation on validation set
val_loss, val_acc, val_prec, val_rec, val_f1, val_preds, val_labels = validate(
    model, val_loader, criterion, device
)

print("=" * 60)
print("FINAL VALIDATION RESULTS")
print("=" * 60)
print(f"Loss:     {val_loss:.4f}")
print(f"Accuracy: {val_acc*100:.2f}%")
print(f"Precision: {val_prec:.4f}")
print(f"Recall:    {val_rec:.4f}")
print(f"F1 Score:  {val_f1:.4f}")
print()

val_cm = confusion_matrix(val_labels, val_preds)
print("Confusion Matrix:")
print(f"              Predicted")
print(f"              Failure  Success")
print(f"Actual Failure  {val_cm[0,0]:6d}  {val_cm[0,1]:6d}")
print(f"        Success  {val_cm[1,0]:6d}  {val_cm[1,1]:6d}")
print()
print("Classification Report:")
print(classification_report(val_labels, val_preds, target_names=['Failure', 'Success']))


## Step 11: Save Model


In [ ]:
# Get checkpoint path from config
import os
from pathlib import Path

checkpoint_path = cfg['data.checkpoint_path']
checkpoint_dir = Path(checkpoint_path).parent
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Save the trained model
torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'input_dim': input_dim,
        'hidden_dims': hidden_dims,
        'output_dim': output_dim,
        'dropout': dropout
    },
    'class_weights': class_weights,
    'best_val_f1': best_val_f1,
    'num_epochs': num_epochs,
    'feature_max': feature_max,
}, checkpoint_path)

print(f"✓ Model saved to {checkpoint_path}")


## Step 12: Load and Evaluate Test Data


In [ ]:
# Load test data directly from roa_labels.txt
# roa_labels.txt already contains individual datapoints (one per row)
# Format: [x, theta, x_dot, theta_dot, label] (5 columns)
# We need to transform to [x, sin(theta), cos(theta), x_dot, theta_dot] for the model
from pathlib import Path

data_dir = Path(cfg['data.data_dir'])
labels_file = data_dir / "roa_labels.txt"

test_start_index = train_size + val_size  # Skip first train_size+val_size rows (used for training/validation)

# Load roa_labels.txt - each row is already an individual datapoint
labels_data = np.loadtxt(labels_file, delimiter=',')

# Get test data (rows test_start_index onwards)
test_data = labels_data[test_start_index:]

print(f"Loading test data from roa_labels.txt (starting from row {test_start_index})...")
print(f"Total rows in roa_labels.txt: {len(labels_data)}")
print(f"Test datapoints: {len(test_data)}")

# Extract features and labels
# Features: columns 0-3 are [x, theta, x_dot, theta_dot]
# Label: column 4 (last column)
test_features_raw = test_data[:, :4]  # [x, theta, x_dot, theta_dot]
y_test = test_data[:, 4].astype(int)  # Labels

# Transform features: [x, theta, x_dot, theta_dot] -> [x, sin(theta), cos(theta), x_dot, theta_dot]
X_test = np.zeros((len(test_features_raw), 5), dtype=np.float32)
X_test[:, 0] = test_features_raw[:, 0]  # x
X_test[:, 1] = np.sin(test_features_raw[:, 1])  # sin(theta)
X_test[:, 2] = np.cos(test_features_raw[:, 1])  # cos(theta)
X_test[:, 3] = test_features_raw[:, 2]  # x_dot
X_test[:, 4] = test_features_raw[:, 3]  # theta_dot

# Normalize test data using the same max values from training
# Only normalize features [0], [3], [4] (x, x_dot, theta_dot)
# Do NOT normalize features [1], [2] (sin(theta), cos(theta))
print(f"\nNormalizing test data using training max values...")
print(f"Feature max values: {feature_max}")
print(f"Normalizing only features [0, 3, 4] (x, x_dot, theta_dot)")
X_test[:, [0, 3, 4]] = X_test[:, [0, 3, 4]] / feature_max[[0, 3, 4]]

print(f"\n✓ Test data loaded: {len(X_test)} datapoints")
print(f"  X_test shape: {X_test.shape}  (datapoints, features)")
print(f"  y_test shape: {y_test.shape}  (labels)")
print(f"  X_test normalized range: [{X_test.min():.4f}, {X_test.max():.4f}]")
print(f"  Success rate: {np.mean(y_test == 1):.2%}")
print(f"  Failure rate: {np.mean(y_test == 0):.2%}")


## Step 13: Test Set Evaluation with Threshold-based Classification


In [ ]:
# Get threshold from config (or use defaults)
failure_threshold = cfg.get('analysis.confidence_threshold', 0.1)
success_threshold = 1.0 - failure_threshold  # Symmetric: 0.9 if failure_threshold is 0.1

# Create test dataset and dataloader
test_dataset = TimestepDataset(X_test, y_test)
test_loader = DataLoader(
    test_dataset, 
    batch_size=batch_size, 
    shuffle=False,
    num_workers=0
)

model.eval()
all_test_preds = []
all_test_labels = []
all_test_probs = []  # Store probabilities for analysis

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        
        outputs = model(batch_X)
        
        # Apply softmax to get probabilities
        probs = torch.softmax(outputs, dim=1)
        prob_success = probs[:, 1].cpu().numpy()  # Probability of success (class 1)
        
        # Threshold-based classification
        # < failure_threshold: failure (0), > success_threshold: success (1), else: uncertain (-1)
        predicted = np.where(prob_success < failure_threshold, 0, 
                    np.where(prob_success > success_threshold, 1, -1))
        
        all_test_preds.extend(predicted)
        all_test_labels.extend(batch_y.cpu().numpy())
        all_test_probs.extend(prob_success)

all_test_preds = np.array(all_test_preds)
all_test_labels = np.array(all_test_labels)
all_test_probs = np.array(all_test_probs)

# Count uncertain predictions
uncertain_mask = (all_test_preds == -1)
num_uncertain = np.sum(uncertain_mask)

print(f"Threshold-based classification:")
print(f"  Failure threshold: < {failure_threshold}")
print(f"  Success threshold: > {success_threshold}")
print(f"  Uncertain zone ({failure_threshold}-{success_threshold}): {num_uncertain} samples ({num_uncertain/len(all_test_preds)*100:.2f}%)")
print()

# Remove uncertain predictions and calculate metrics
valid_mask = (all_test_preds != -1)
all_test_preds_filtered = all_test_preds[valid_mask]
all_test_labels_filtered = all_test_labels[valid_mask]

test_accuracy = accuracy_score(all_test_labels_filtered, all_test_preds_filtered)
test_precision = precision_score(all_test_labels_filtered, all_test_preds_filtered)
test_recall = recall_score(all_test_labels_filtered, all_test_preds_filtered)
test_f1 = f1_score(all_test_labels_filtered, all_test_preds_filtered)
test_cm = confusion_matrix(all_test_labels_filtered, all_test_preds_filtered)

print("=" * 60)
print("TEST SET RESULTS (Threshold-based Classification)")
print("=" * 60)
print(f"Total test samples: {len(all_test_labels)}")
print(f"Uncertain samples removed: {num_uncertain}")
print(f"Remaining samples: {len(all_test_labels_filtered)}")
print(f"Accuracy: {test_accuracy*100:.2f}%")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1 Score:  {test_f1:.4f}")
print()
print("Confusion Matrix:")
print(f"              Predicted")
print(f"              Failure  Success")
print(f"Actual Failure  {test_cm[0,0]:6d}  {test_cm[0,1]:6d}")
print(f"        Success  {test_cm[1,0]:6d}  {test_cm[1,1]:6d}")
print()
print("Classification Report:")
print(classification_report(all_test_labels_filtered, all_test_preds_filtered, target_names=['Failure', 'Success']))
